# Property Feature Preprocessing Notebook

In [1]:
from __future__ import annotations

import sys

sys.path.insert(0, '..')

from pathlib import Path
import numpy as np
import pandas as pd

from specs import ATTACK_SPECS

from utils.preprocessing import (
    FEATURES,
    PROPERTY_BOOLEAN_FEATURES
)

In [2]:
DATASET_NAME = "ciciot2023"
# DATASET_NAME = "cicids2017"
DATASET_TO_LOAD = "preprocessed"
INPUT_PATH = f"../data/{DATASET_NAME}_{DATASET_TO_LOAD}.tsv"
OUTPUT_PATH = f"./{DATASET_NAME}_{DATASET_TO_LOAD}_with_properties.tsv"

WINDOW_SECONDS = 5.0
TARGET_LABELS = ["BENIGN", "DOS_HTTP_FLOOD", "PORTSCAN", "ATTACK"]

In [3]:
def normalize_label_name(label: str) -> str:
    label = str(label).strip()
    return (
        label.replace("-", "_")
        .replace(" ", "_")
        .replace("/", "_")
        .upper()
    )


def compute_time_elapsed(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    required = {"id.orig_h", "id.resp_h", "ts"}
    if not required.issubset(df.columns):
        df["time_elapsed"] = 999999.0
        return df

    df = df.sort_values(["id.orig_h", "id.resp_h", "ts"]).reset_index(drop=True)
    df["time_elapsed"] = (
        df.groupby(["id.orig_h", "id.resp_h"])["ts"]
        .diff()
        .fillna(999999.0)
    )
    return df


def compute_valid_tcp_handshake(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    proto_ok =  df["proto"].eq("tcp")
    history_ok = df["history"].str.contains(r"S.*h.*A", regex=True, na=False)
    state_ok = df["conn_state"].isin(["S1", "SF", "RSTO"])

    df["valid_tcp_handshake"] = (proto_ok & history_ok & state_ok).astype(int)
    return df

def compute_valid_http_conn(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    proto_ok =  df["proto"].eq("tcp")
    port_http = df["id.resp_p"].isin([80, 8080, 8000])
    port_https = df["id.resp_p"].isin([443, 8443])
    service_http = df["service"].str.lower().eq("http")
    service_https = df["service"].str.lower().eq("https")
    service_ssl = df["service"].str.lower().eq("ssl")

    df["valid_http_conn"] = (
        proto_ok &
        (
            (port_http & service_http) |
            (port_https & (service_https | service_ssl))
        )
    ).astype(int)
    return df


def add_shared_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    numeric_cols = [
        "ts",
        "duration",
        "orig_bytes",
        "resp_bytes",
        "missed_bytes",
        "orig_pkts",
        "orig_ip_bytes",
        "resp_pkts",
        "resp_ip_bytes",
        "id.resp_p",
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    existing_numeric = [c for c in numeric_cols if c in df.columns]
    if existing_numeric:
        df[existing_numeric] = df[existing_numeric].replace([np.inf, -np.inf], np.nan)
        df[existing_numeric] = df[existing_numeric].fillna(0.0)

    df = compute_time_elapsed(df)
    df = compute_valid_tcp_handshake(df)
    df = compute_valid_http_conn(df)

    duration_safe = df["duration"].copy() if "duration" in df.columns else pd.Series(1e-6, index=df.index)
    duration_safe = duration_safe.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    duration_safe = duration_safe.mask(duration_safe <= 0, 1e-6)

    df["orig_pkt_rate"] = df["orig_pkts"] / duration_safe if "orig_pkts" in df.columns else 0.0
    df["orig_byte_rate"] = df["orig_bytes"] / duration_safe if "orig_bytes" in df.columns else 0.0
    df["flood_rate"] = df["orig_bytes"] / duration_safe if "orig_bytes" in df.columns else 0.0
    df["pkt_asymmetry"] = df["orig_pkts"] / (df["resp_pkts"] + 1.0)
    df["byte_asymmetry"] = df["orig_bytes"] / (df["resp_bytes"] + 1.0)

    df["is_tcp"] = (
        df["proto"].astype(str).str.lower().eq("tcp").astype(int)
    )

    engineered_cols = [
        "orig_pkt_rate",
        "orig_byte_rate",
        "pkt_asymmetry",
        "byte_asymmetry",
        "time_elapsed",
        "flood_rate",
        "valid_tcp_handshake",
        "is_tcp",
        "valid_http_conn",
    ]

    for col in engineered_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)

    return df


def add_portscan_context_features(df: pd.DataFrame, window_seconds: float = 5.0) -> pd.DataFrame:
    df = df.copy()

    required = {"ts", "id.orig_h", "id.resp_p", "orig_pkts", "duration", "conn_state"}
    if not required.issubset(df.columns):
        for col in ["uniq_dst_ports", "pkts_per_port", "scan_duration", "fail_ratio"]:
            df[col] = 0.0
        return df

    df["window_id"] = (df["ts"] // window_seconds).astype(int)
    group_cols = ["id.orig_h", "window_id"]

    agg = df.groupby(group_cols).agg(
        uniq_dst_ports=("id.resp_p", "nunique"),
        total_orig_pkts=("orig_pkts", "sum"),
        scan_duration=("duration", "max"),
        total_flows=("id.orig_h", "size"),
    ).reset_index()

    failed_states = {"S0", "REJ", "RSTO", "RSTR", "RSTOS0", "RSTRH", "SH", "SHR"}
    df["is_failed_conn"] = df["conn_state"].astype(str).isin(failed_states).astype(int)

    fail_agg = df.groupby(group_cols).agg(
        failed_flows=("is_failed_conn", "sum"),
    ).reset_index()

    agg = agg.merge(fail_agg, on=group_cols, how="left")
    agg["failed_flows"] = agg["failed_flows"].fillna(0.0)

    agg["pkts_per_port"] = agg["total_orig_pkts"] / agg["uniq_dst_ports"].clip(lower=1)
    agg["fail_ratio"] = agg["failed_flows"] / agg["total_flows"].clip(lower=1)

    df = df.merge(
        agg[
            [
                "id.orig_h",
                "window_id",
                "uniq_dst_ports",
                "pkts_per_port",
                "scan_duration",
                "fail_ratio",
            ]
        ],
        on=["id.orig_h", "window_id"],
        how="left",
    )

    for col in ["uniq_dst_ports", "pkts_per_port", "scan_duration", "fail_ratio"]:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

    return df


def add_property_boolean_features(df: pd.DataFrame, specs: dict) -> pd.DataFrame:
    df = df.copy()

    validity = specs["validity"]
    dos_http = specs["dos_http_flood"]
    portscan = specs["portscan"]

    needed_for_valid_input = [
        c for c in [
            "duration",
            "orig_bytes",
            "resp_bytes",
            "orig_pkts",
            "orig_pkt_rate",
            "time_elapsed",
            "flood_rate",
        ] if c in df.columns
    ]

    if needed_for_valid_input:
        df["valid_input"] = np.isfinite(df[needed_for_valid_input]).all(axis=1).astype(int)
    else:
        df["valid_input"] = 1.0

    df["valid_duration"] = (
        (df["duration"] >= dos_http["valid_duration_min"]) &
        (df["duration"] <= dos_http["valid_duration_max"])
    ).astype(int)

    avg_orig_pkt_size = df["orig_bytes"] / np.maximum(df["orig_pkts"], 1.0)
    df["valid_packet_size"] = (
        (df["orig_pkts"] >= validity["valid_packet_size_min_pkts"]) &
        (avg_orig_pkt_size >= validity["valid_packet_size_min_avg_bytes"]) &
        (df["orig_bytes"] >= validity["valid_packet_size_min_total_bytes"])
    ).astype(int)

    df["valid_iat"] = (df["orig_pkt_rate"] <= dos_http["valid_iat_max_pkt_rate"]).astype(int)

    df["dos_http_mal_time_elapsed"] = (df["time_elapsed"] <= dos_http["mal_time_elapsed_max"]).astype(int)
    df["dos_http_mal_flood_rate"] = (df["flood_rate"] >= dos_http["mal_flood_rate_min"]).astype(int)

    df["portscan_many_ports"] = (df["uniq_dst_ports"] >= portscan["many_ports_min"]).astype(int)
    df["portscan_few_pkts_per_port"] = (df["pkts_per_port"] <= portscan["few_pkts_per_port_max"]).astype(int)
    df["portscan_short_duration"] = (df["scan_duration"] <= portscan["short_scan_duration_max"]).astype(int)
    df["portscan_high_fail_ratio"] = (df["fail_ratio"] >= portscan["high_fail_ratio_min"]).astype(int)

    return df


def load_and_prepare_dataset(
    input_path: str,
    target_labels: list[str] | None = None,
    window_seconds: float = 5.0,
) -> pd.DataFrame:
    print(f"Loading data from: {input_path}")
    df = pd.read_csv(input_path, delimiter="\t", on_bad_lines="skip")
    df.columns = df.columns.str.strip()

    print("Raw shape:", df.shape)

    df.drop_duplicates(inplace=True)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    if "label" in df.columns:
        df["label"] = df["label"].astype(str).map(normalize_label_name)
        if target_labels is not None:
            target_labels = [normalize_label_name(x) for x in target_labels]
            df = df[df["label"].isin(target_labels)].copy()

    df = add_shared_engineered_features(df)
    df = add_portscan_context_features(df, window_seconds=window_seconds)
    df = add_property_boolean_features(df, ATTACK_SPECS)

    print("Processed shape:", df.shape)
    if "label" in df.columns:
        print(df["label"].value_counts())

    return df


In [4]:
processed_df = load_and_prepare_dataset(
    input_path=INPUT_PATH,
    target_labels=TARGET_LABELS,
    window_seconds=WINDOW_SECONDS,
)

processed_df.head()

Loading data from: ../data/ciciot2023_preprocessed.tsv
Raw shape: (4285835, 23)
Processed shape: (3254459, 48)
label
DOS_HTTP_FLOOD    1508589
ATTACK            1187082
BENIGN             342255
PORTSCAN           216533
Name: count, dtype: int64


,ts,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,service,duration,orig_bytes,...,valid_input,valid_duration,valid_packet_size,valid_iat,dos_http_mal_time_elapsed,dos_http_mal_flood_rate,portscan_many_ports,portscan_few_pkts_per_port,portscan_short_duration,portscan_high_fail_ratio
0,1.664843e+09,CmdFgx1dhU9S5BTltf,0.0.0.0,49309,192.168.137.1,53,udp,dns,0.000000,0.0,...,1,1,0,0,0,0,0,1,1,1
1,1.659989e+09,C9oHjs4ERb2JSmEg4h,0.0.0.0,0,224.0.0.22,0,unknown_transport,-,5.060887,0.0,...,1,1,0,1,0,0,0,1,0,1
2,1.659989e+09,C5Q48JW4vSaxDBdGf,0.0.0.0,5353,224.0.0.251,5353,udp,dns,0.000000,0.0,...,1,1,0,0,0,0,0,1,0,1
3,1.659976e+09,Ck4L27ZsL1CZshHm1,0.0.0.0,68,255.255.255.255,67,udp,dhcp,700.529673,47349.0,...,1,0,1,1,0,0,0,0,0,1
4,1.659984e+09,CZr3xz1BJZRqdXbNge,0.0.0.0,68,255.255.255.255,67,udp,dhcp,591.069092,41078.0,...,1,0,1,1,0,0,0,0,0,1


In [5]:
processed_df.to_csv(OUTPUT_PATH, sep="\t", index=False)
print(f"Saved processed dataset to: {OUTPUT_PATH}")

Saved processed dataset to: ./ciciot2023_preprocessed_with_properties.tsv


In [6]:
print("\nBoolean property feature means:")
processed_df[PROPERTY_BOOLEAN_FEATURES].mean().sort_values()


Boolean property feature means:


valid_tcp_handshake           0.020838
portscan_many_ports           0.052066
valid_http_conn               0.070172
portscan_few_pkts_per_port    0.118516
dos_http_mal_flood_rate       0.391369
valid_packet_size             0.395548
portscan_short_duration       0.506330
portscan_high_fail_ratio      0.533873
valid_duration                0.672725
dos_http_mal_time_elapsed     0.866640
is_tcp                        0.871871
valid_iat                     0.929055
valid_input                   1.000000
dtype: float64

In [7]:
processed_df[processed_df["label"] == "BENIGN"][PROPERTY_BOOLEAN_FEATURES].mean().sort_values()

portscan_many_ports           0.001291
valid_tcp_handshake           0.025922
valid_http_conn               0.166037
is_tcp                        0.219746
portscan_high_fail_ratio      0.230556
dos_http_mal_time_elapsed     0.271274
portscan_few_pkts_per_port    0.391506
dos_http_mal_flood_rate       0.406936
valid_packet_size             0.425323
portscan_short_duration       0.699747
valid_iat                     0.876253
valid_duration                0.970133
valid_input                   1.000000
dtype: float64

In [8]:
processed_df[processed_df["label"] == "DOS_HTTP_FLOOD"][PROPERTY_BOOLEAN_FEATURES].mean().sort_values()

portscan_many_ports           0.000057
portscan_few_pkts_per_port    0.007159
valid_tcp_handshake           0.025027
dos_http_mal_flood_rate       0.065637
valid_http_conn               0.090001
valid_packet_size             0.092440
portscan_short_duration       0.720147
valid_iat                     0.942940
portscan_high_fail_ratio      0.960118
dos_http_mal_time_elapsed     0.978539
is_tcp                        0.982318
valid_duration                0.985901
valid_input                   1.000000
dtype: float64

# Create balanced datasets with properties

In [9]:
from sklearn.model_selection import train_test_split
from utils.preprocessing import balance_df

In [10]:
df_copy = processed_df.copy()
X = df_copy.drop(columns=["label"]).copy()
y = df_copy["label"].copy()

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    stratify=y,
    random_state=42,
)

In [12]:
train = X_train.copy()
train["label"] = y_train

In [13]:
train_bal = balance_df(train)

Balancing dataset to 151573 rows per class


In [14]:
train_bal.to_csv(f"./{DATASET_NAME}_{DATASET_TO_LOAD}_with_properties_train_balanced.tsv", sep="\t", index=False)

In [15]:
# Save test sets as well for later evaluation
X_test["label"] = y_test.values
X_test.to_csv(f"{DATASET_NAME}_{DATASET_TO_LOAD}_with_properties_test.tsv", sep='\t', index=False, header=True)

In [16]:
train_bal_small = balance_df(train, frac=0.1)
train_bal_small.to_csv(f"./{DATASET_NAME}_{DATASET_TO_LOAD}_with_properties_train_balanced_small.tsv", sep='\t', index=False, header=True)


Balancing dataset to 15157 rows per class


# Generate filtered dataset with only valid attack rows

In [40]:
df_copy = processed_df.copy()
df_copy = df_copy[df_copy["label"] == "DOS_HTTP_FLOOD"].copy()

In [46]:
def drop_rejected_http_flood_rows(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    rejected_states = {
        "REJ", "S0", "OTH"
    }


    mask = ~(
        (df["label"] == "DOS_HTTP_FLOOD") &
        (df["conn_state"].astype(str).str.upper().isin(rejected_states)) |
        (df["proto"].astype(str).str.lower().ne("tcp"))
    )

    removed = (~mask).sum()
    print(f"Removed {removed} DOS_HTTP_FLOOD rows (REJ or non-TCP)")

    return df.loc[mask].reset_index(drop=True)

In [47]:
df_copy = drop_rejected_http_flood_rows(df_copy)

Removed 21152 DOS_HTTP_FLOOD rows (REJ or non-TCP)


In [48]:
df_copy[df_copy["label"] == "DOS_HTTP_FLOOD"][PROPERTY_BOOLEAN_FEATURES].mean().sort_values()

portscan_many_ports           0.000068
portscan_few_pkts_per_port    0.003937
portscan_short_duration       0.151178
valid_tcp_handshake           0.198695
dos_http_mal_flood_rate       0.448296
valid_http_conn               0.625382
valid_packet_size             0.681257
portscan_high_fail_ratio      0.889777
valid_duration                0.900024
dos_http_mal_time_elapsed     0.943494
valid_iat                     0.997158
valid_input                   1.000000
is_tcp                        1.000000
dtype: float64

In [50]:
df_copy.to_csv(f"./{DATASET_NAME}_{DATASET_TO_LOAD}_with_properties_no_rej_http_flood.tsv", sep='\t', index=False, header=True)